# FolkIn — Adaptasi Qwen menggunakan LoRA

Notebook ini melakukan demonstrasi parameter-efficient fine-tuning pada Qwen2.5-0.5B-Instruct. Dataset contoh berisi respons fokus sintetis untuk prototype, bukan dataset percakapan produksi.

Jalankan di Google Colab dengan GPU. Model dasar tetap pretrained; kita tidak melatih LLM dari nol.


In [ ]:
!pip -q install -U transformers peft accelerate datasets


## 1. Unggah dataset instruksi

Unggah file `qwen_focus_examples.jsonl` dari starter kit.


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
import os, json
from datasets import Dataset

jsonl_name = next((n for n in uploaded if n.endswith(".jsonl")), None)
if jsonl_name is None:
    raise FileNotFoundError("Unggah qwen_focus_examples.jsonl terlebih dahulu.")

rows = []
with open(jsonl_name, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

print("Jumlah contoh:", len(rows))
print(rows[0])
dataset = Dataset.from_list(rows).train_test_split(test_size=0.1, seed=42)


## 2. Muat Qwen pretrained dan siapkan LoRA

Jika GPU tidak tersedia, pindahkan runtime ke GPU. Model 0.5B dipilih agar eksperimen lebih ringan.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 3. Tokenisasi data

Loss training difokuskan pada jawaban assistant. Contoh ini sengaja dibatasi panjangnya agar sesuai untuk eksperimen awal.


In [ ]:
MAX_LENGTH = 512

def preprocess(example):
    user_text = example["instruction"]
    if example.get("input"):
        user_text += "\\nKonteks: " + example["input"]

    prompt_messages = [
        {"role": "system", "content": "Kamu adalah FolkIn, asisten fokus belajar yang suportif, ringkas, dan tidak menghakimi. Jangan mengklaim bahwa deteksi webcam selalu benar."},
        {"role": "user", "content": user_text}
    ]
    prompt = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    full_text = prompt + example["output"] + tokenizer.eos_token

    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full = tokenizer(
        full_text, max_length=MAX_LENGTH, truncation=True,
        padding=False, add_special_tokens=False
    )
    labels = full["input_ids"].copy()
    prompt_len = min(len(prompt_ids), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    full["labels"] = labels
    return full

tokenized = dataset.map(preprocess, remove_columns=dataset["train"].column_names)
print(tokenized)


## 4. Fine-tuning LoRA

Tiga epoch hanya titik awal. Evaluasi respons sebelum menyimpulkan bahwa model sudah lebih baik.


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="/content/folkin_qwen_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    remove_unused_columns=False
)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=collator
)
trainer.train()

ADAPTER_DIR = "/content/folkin_qwen_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter disimpan di", ADAPTER_DIR)


## 5. Uji inference

Bandingkan hasil beberapa prompt sebelum dan sesudah fine-tuning. Respons tetap perlu diperiksa manusia.


In [ ]:
def ask_folkin(instruction, context=""):
    user_text = instruction + (("\\nKonteks: " + context) if context else "")
    messages = [
        {"role": "system", "content": "Kamu adalah FolkIn, asisten fokus belajar yang suportif, ringkas, dan tidak menghakimi."},
        {"role": "user", "content": user_text}
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=100, do_sample=True,
            temperature=0.7, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    answer = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return answer.strip()

print(ask_folkin(
    "Buat peringatan singkat ketika HP terdeteksi sedang digenggam.",
    "Tugas pengguna: menyelesaikan laporan praktikum. Deteksi stabil: 3 detik."
))


## 6. Unduh adapter LoRA

Adapter ini bukan model Qwen lengkap. Untuk menggunakannya, aplikasi memuat model dasar Qwen2.5-0.5B-Instruct dan adapter LoRA ini.


In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/folkin_qwen_adapter", "zip", ADAPTER_DIR)
files.download("/content/folkin_qwen_adapter.zip")
